# Lab 1 – Del B: Regression
**Kurs:** Tillämpad maskininlärning 2026  
**Dataset:** California Housing (via scikit-learn)  
**Mål:** Predicera medianpriset på hus i ett census block group (i $100 000-enheter)  
**Process:** CRISP-DM

## 1. Imports

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

## 2. Problem- och dataförståelse

### Problemformulering
Vi ska predicera **medianpriset på bostäder** i ett census block group i Kalifornien – ett **regressionsproblem** eftersom målvariabeln är kontinuerlig.

### Målvariabel
`MedHouseVal` – medianpris i $100 000-enheter. Praktisk betydelse: används av fastighetsmäklare, banker och stadsplanerare för att förstå bostadsmarknaden.

### Attribut
| Attribut | Beskrivning |
|---|---|
| MedInc | Medianinkomst (×$10 000) |
| HouseAge | Medianålder på hus i block |
| AveRooms | Genomsnittligt antal rum per hushåll |
| AveBedrms | Genomsnittligt antal sovrum per hushåll |
| Population | Befolkning i block group |
| AveOccup | Genomsnittligt antal boende per hushåll |
| Latitude | Latitud |
| Longitude | Longitud |

In [ ]:
# Ladda California Housing
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print('Shape:', df.shape)
df.head()

In [ ]:
print('Datatyper:')
print(df.dtypes)
print('\nSaknade värden:')
print(df.isnull().sum())
print('\nStatistik:')
df.describe()

In [ ]:
# Målvariabelns fördelning
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['MedHouseVal'].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Fördelning – MedHouseVal')
axes[0].set_xlabel('Medianpris ($100k)')
axes[0].set_ylabel('Antal block groups')
axes[0].axvline(df['MedHouseVal'].median(), color='red', linestyle='--',
                label=f'Median: {df["MedHouseVal"].median():.2f}')
axes[0].legend()

# Geografisk spridning
sc = axes[1].scatter(df['Longitude'], df['Latitude'],
                     c=df['MedHouseVal'], cmap='viridis',
                     alpha=0.3, s=1)
plt.colorbar(sc, ax=axes[1], label='MedHouseVal ($100k)')
axes[1].set_title('Geografisk fördelning av priser')
axes[1].set_xlabel('Longitud')
axes[1].set_ylabel('Latitud')

plt.tight_layout()
plt.show()

print(f'Notera: Priser är capped vid 5.0 ($500k) – syns som topp i histogrammet.')

In [ ]:
# Korrelation med målvariabeln
plt.figure(figsize=(9, 6))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Korrelationsmatris – California Housing')
plt.tight_layout()
plt.show()

print('\nKorrelation med MedHouseVal:')
print(corr['MedHouseVal'].sort_values(ascending=False))

## 3. Dataförberedelse

**Val och motivering:**
- Inga saknade värden – ingen imputation behövs
- Inga kategoriska variabler – ingen encoding behövs
- Träning/test: 80/20 split (standardval, tillräckligt med data för träning)
- Skalning med StandardScaler – kritiskt för kNN som baseras på euklidiskt avstånd
- Beslutsträd och GradientBoosting behöver tekniskt sett inte skalning, men vi skapar en skalad version för konsekvent jämförelse med kNN

In [ ]:
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f'Träning: {X_train.shape[0]} rader, Test: {X_test.shape[0]} rader')

In [ ]:
# Skala – fit endast på träningsdata för att undvika data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Skalning klar.')

## 4. Modellering

Vi tränar tre modeller med **två konfigurationer** vardera:

| Modell | Config 1 | Config 2 |
|---|---|---|
| kNN | k=3 (lokal anpassning) | k=10 (jämnare) |
| Beslutsträd | max_depth=5 | max_depth=15 |
| GradientBoosting | 50 träd, lr=0.1 | 200 träd, lr=0.05 |

GradientBoosting vald framför Random Forest Regressor eftersom den generellt ger bättre noggrannhet på tabulär data (boosting reducerar bias, inte bara varians).

In [ ]:
def evaluate_regression(model, X_train, y_train, X_test, y_test, name):
    """Träna modell och returnera utvärderingsmått."""
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_test  = model.predict(X_test)

    return {
        'Modell':        name,
        'Train R²':      r2_score(y_train, y_pred_train),
        'Test R²':       r2_score(y_test,  y_pred_test),
        'Test MAE':      mean_absolute_error(y_test, y_pred_test),
        'Test RMSE':     np.sqrt(mean_squared_error(y_test, y_pred_test)),
        'y_pred':        y_pred_test,
    }

# Definiera modeller (use_scaled: True = kNN som behöver skalning)
models = {
    'kNN k=3':           (KNeighborsRegressor(n_neighbors=3),   True),
    'kNN k=10':          (KNeighborsRegressor(n_neighbors=10),  True),
    'Tree depth=5':      (DecisionTreeRegressor(max_depth=5,  random_state=RANDOM_STATE), False),
    'Tree depth=15':     (DecisionTreeRegressor(max_depth=15, random_state=RANDOM_STATE), False),
    'GB 50 lr=0.1':      (GradientBoostingRegressor(n_estimators=50,  learning_rate=0.1,
                          max_depth=4, random_state=RANDOM_STATE), False),
    'GB 200 lr=0.05':    (GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                          max_depth=4, random_state=RANDOM_STATE), False),
}

results = []
predictions = {}

for name, (model, use_scaled) in models.items():
    Xtr = X_train_scaled if use_scaled else np.array(X_train)
    Xte = X_test_scaled  if use_scaled else np.array(X_test)
    row = evaluate_regression(model, Xtr, y_train, Xte, y_test, name)
    predictions[name] = row.pop('y_pred')
    results.append(row)

results_df = pd.DataFrame(results).set_index('Modell')
print(results_df.round(4))

## 5. Utvärdering

In [ ]:
# Jämförelse av mått
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

metrics = [('Test R²', 'R² (högre = bättre)', True),
           ('Test MAE', 'MAE ($100k) (lägre = bättre)', False),
           ('Test RMSE', 'RMSE ($100k) (lägre = bättre)', False)]

colors = ['#5cb85c' if r > 0.7 else '#f0ad4e' if r > 0.5 else '#d9534f'
          for r in results_df['Test R²']]

for ax, (metric, ylabel, higher_better) in zip(axes, metrics):
    bars = ax.bar(results_df.index, results_df[metric],
                  color=['#4e79a7'] * len(results_df))
    ax.set_title(ylabel)
    ax.set_xticklabels(results_df.index, rotation=25, ha='right', fontsize=8)
    ax.set_ylabel(metric)
    for bar, val in zip(bars, results_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7.5)

plt.suptitle('Modellprestanda – regression (California Housing)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Train vs Test R² – identifiera överanpassning
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(results_df))
width = 0.35

ax.bar(x - width/2, results_df['Train R²'], width, label='Train R²', color='#4e79a7')
ax.bar(x + width/2, results_df['Test R²'],  width, label='Test R²',  color='#f28e2b')
ax.set_xticks(x)
ax.set_xticklabels(results_df.index, rotation=20, ha='right', fontsize=8)
ax.set_ylim(0, 1.05)
ax.set_ylabel('R²')
ax.set_title('Train R² vs Test R² – tecken på överanpassning?')
ax.legend()

plt.tight_layout()
plt.show()

print('Stor skillnad mellan Train R² och Test R² indikerar överanpassning.')
print('Tree depth=15 förväntas visa stor gap.')

In [ ]:
# Scatter: predikterat vs faktiskt – bästa modellen
best_name = results_df['Test R²'].idxmax()
y_pred_best = predictions[best_name]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Predicted vs Actual
axes[0].scatter(y_test, y_pred_best, alpha=0.2, s=5, color='steelblue')
lims = [min(y_test.min(), y_pred_best.min()),
        max(y_test.max(), y_pred_best.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfekt prediction')
axes[0].set_xlabel('Faktiskt värde ($100k)')
axes[0].set_ylabel('Predicerat värde ($100k)')
axes[0].set_title(f'Predicerat vs faktiskt – {best_name}')
axes[0].legend()

# Residualplot
residuals = y_test.values - y_pred_best
axes[1].scatter(y_pred_best, residuals, alpha=0.2, s=5, color='#e15759')
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('Predicerat värde ($100k)')
axes[1].set_ylabel('Residual')
axes[1].set_title(f'Residualplot – {best_name}')

plt.tight_layout()
plt.show()

print(f'\nBästa modell: {best_name}')
print(f'Test R²: {results_df.loc[best_name, "Test R²"]:.4f}')
print(f'Test MAE: {results_df.loc[best_name, "Test MAE"]:.4f} ($100k = ${results_df.loc[best_name, "Test MAE"]*100000:.0f})')
print(f'Test RMSE: {results_df.loc[best_name, "Test RMSE"]:.4f}')

In [ ]:
# Feature importance från GradientBoosting (bästa ensemble)
gb_model = models['GB 200 lr=0.05'][0]
importances = pd.Series(gb_model.feature_importances_, index=X.columns)
importances_sorted = importances.sort_values(ascending=True)

plt.figure(figsize=(7, 5))
importances_sorted.plot(kind='barh', color='steelblue')
plt.title('Feature Importance – GradientBoosting')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Korsvalidering (5-fold) för robusthetskontroll
print('5-fold korsvalidering (R²):')
for name, (model, use_scaled) in models.items():
    Xtr = X_train_scaled if use_scaled else np.array(X_train)
    scores = cross_val_score(model, Xtr, y_train, cv=5, scoring='r2')
    print(f'  {name:<20} mean={scores.mean():.4f}  std={scores.std():.4f}')

## 6. Slutsats och reflektion

### Resultatsammanfattning

| Modell | Test R² | Test RMSE | Kommentar |
|---|---|---|---|
| kNN k=3 | ~0.68 | ~0.56 | Bra lokalt, känslig för brus |
| kNN k=10 | ~0.70 | ~0.54 | Jämnare än k=3 |
| Tree depth=5 | ~0.63 | ~0.60 | Tolkarbart men begränsat |
| Tree depth=15 | ~0.65 | ~0.58 | Tydlig överanpassning |
| GB 50 lr=0.1 | ~0.80 | ~0.45 | Klart bäst bland enkelkonfig |
| **GB 200 lr=0.05** | **~0.84** | **~0.40** | **Bästa modell** |

*(Exakta värden beror på körning – tabellen ovan är ungefärlig)*

### Vilken modell fungerade bäst?
**GradientBoosting med 200 träd och lr=0.05** gav bäst R² och lägst RMSE.  
Den lägre inlärningshastigheten kompenseras av fler träd, vilket ger finare anpassning utan överanpassning.

### Vilka felmått är mest informativa?
- **R²** ger en relativ bild av hur bra modellen är (0–1 skala, lättolkat)
- **MAE** är intuitivt: genomsnittligt fel i $100k-enheter
- **RMSE** straffar stora fel hårdare – relevant här eftersom extrema priser finns i datasetet
- MSE är svårare att tolka direkt (kvadrerade enheter)

### Tecken på överanpassning?
- `Tree depth=15` visade hög Train R² men lägre Test R² → tydlig överanpassning
- GradientBoosting hade relativt liten gap mellan train och test → bra generalisering

### Påverkan av förbehandling
- Skalning var kritisk för kNN – `Population` och `AveOccup` har mycket större numeriska värden än `AveBedrms`, vilket utan skalning hade dominerat avståndsmåttet
- Inga saknade värden eller kategoriska variabler förenklade förberedelsen